In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sales_silver = spark.table("workspace.default.silver_sales")

print("Silver Sales rows:", sales_silver.count())

Silver Sales rows: 125751


In [0]:
gold_daily_demand = sales_silver.groupBy(
    "transaction_date",
    "product_id",
    "store"
).agg(
    F.sum("net_qty_sold").alias("daily_units_sold"),
    F.sum("sales_amount").alias("daily_sales"),
    F.sum("gross_margin").alias("daily_gross_margin"),
    F.sum("transaction_count").alias("daily_transactions"),
    F.sum(
        F.when(F.col("is_return") == 1, 1).otherwise(0)
    ).alias("return_count")
)

In [0]:
window_7 = Window.partitionBy(
    "product_id", "store"
).orderBy(
    "transaction_date"
).rowsBetween(-6, 0)

window_30 = Window.partitionBy(
    "product_id", "store"
).orderBy(
    "transaction_date"
).rowsBetween(-29, 0)

gold_daily_demand = gold_daily_demand.withColumn(
    "rolling_7d_units",
    F.sum("daily_units_sold").over(window_7)
).withColumn(
    "rolling_30d_units",
    F.sum("daily_units_sold").over(window_30)
).withColumn(
    "avg_daily_demand_7d",
    F.col("rolling_7d_units") / 7
).withColumn(
    "avg_daily_demand_30d",
    F.col("rolling_30d_units") / 30
)

In [0]:
gold_daily_demand.orderBy(
    F.desc("transaction_date")
).show(20, False)

+----------------+-----------+--------+----------------+-----------+------------------+------------------+------------+----------------+-----------------+-------------------+--------------------+
|transaction_date|product_id |store   |daily_units_sold|daily_sales|daily_gross_margin|daily_transactions|return_count|rolling_7d_units|rolling_30d_units|avg_daily_demand_7d|avg_daily_demand_30d|
+----------------+-----------+--------+----------------+-----------+------------------+------------------+------------+----------------+-----------------+-------------------+--------------------+
|2026-04-24      |PROD-100803|STR-1308|1.0             |90.0       |44.45             |1                 |0           |7.0             |10.0             |1.0                |0.3333333333333333  |
|2026-04-24      |PROD-100803|STR-1082|1.0             |90.0       |44.46             |1                 |0           |9.0             |18.0             |1.2857142857142858 |0.6                 |
|2026-04-24      |PR

In [0]:
# Create the complete date range
date_bounds = sales_silver.agg(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]

print("Date range:", min_date, "to", max_date)

Date range: 2025-06-01 to 2026-04-24


In [0]:
# All Product × Store combinations appearing in sales
product_store = sales_silver.select(
    "product_id",
    "store"
).distinct()

# Complete calendar
calendar = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{min_date}'),
            to_date('{max_date}'),
            interval 1 day
        )
    ) AS transaction_date
""")

# Complete Product × Store × Date spine
demand_spine = product_store.crossJoin(calendar)

print("Product-Store combinations:", product_store.count())
print("Calendar days:", calendar.count())
print("Spine rows:", demand_spine.count())

Product-Store combinations: 50447
Calendar days: 328
Spine rows: 16546616


In [0]:
daily_actuals = sales_silver.groupBy(
    "transaction_date",
    "product_id",
    "store"
).agg(
    F.sum("net_qty_sold").alias("daily_units_sold"),
    F.sum("sales_amount").alias("daily_sales"),
    F.sum("gross_margin").alias("daily_gross_margin"),
    F.sum("transaction_count").alias("daily_transactions"),
    F.sum(
        F.when(F.col("is_return") == 1, 1).otherwise(0)
    ).alias("return_count")
)

gold_daily_demand = demand_spine.join(
    daily_actuals,
    ["transaction_date", "product_id", "store"],
    "left"
).fillna({
    "daily_units_sold": 0,
    "daily_sales": 0,
    "daily_gross_margin": 0,
    "daily_transactions": 0,
    "return_count": 0
})

In [0]:
window_7 = Window.partitionBy(
    "product_id", "store"
).orderBy(
    "transaction_date"
).rowsBetween(-6, 0)

window_30 = Window.partitionBy(
    "product_id", "store"
).orderBy(
    "transaction_date"
).rowsBetween(-29, 0)

gold_daily_demand = gold_daily_demand.withColumn(
    "rolling_7d_units",
    F.sum("daily_units_sold").over(window_7)
).withColumn(
    "rolling_30d_units",
    F.sum("daily_units_sold").over(window_30)
).withColumn(
    "avg_daily_demand_7d",
    F.col("rolling_7d_units") / 7
).withColumn(
    "avg_daily_demand_30d",
    F.col("rolling_30d_units") / 30
)

In [0]:
gold_daily_demand.orderBy(
    F.desc("transaction_date")
).show(20, False)

+----------------+-----------+--------+----------------+-----------+------------------+------------------+------------+----------------+-----------------+-------------------+--------------------+
|transaction_date|product_id |store   |daily_units_sold|daily_sales|daily_gross_margin|daily_transactions|return_count|rolling_7d_units|rolling_30d_units|avg_daily_demand_7d|avg_daily_demand_30d|
+----------------+-----------+--------+----------------+-----------+------------------+------------------+------------+----------------+-----------------+-------------------+--------------------+
|2026-04-24      |PROD-100044|STR-1006|0.0             |0.0        |0.0               |0                 |0           |0.0             |0.0              |0.0                |0.0                 |
|2026-04-24      |PROD-100043|STR-1350|0.0             |0.0        |0.0               |0                 |0           |0.0             |0.0              |0.0                |0.0                 |
|2026-04-24      |PR

In [0]:
# Basic validation
print("Gold rows:", gold_daily_demand.count())

print(
    "Distinct Product-Store pairs:",
    gold_daily_demand.select("product_id", "store").distinct().count()
)

print(
    "Date range:",
    gold_daily_demand.agg(
        F.min("transaction_date"),
        F.max("transaction_date")
    ).collect()[0]
)

Gold rows: 16546616
Distinct Product-Store pairs: 50447
Date range: Row(min(transaction_date)=datetime.date(2025, 6, 1), max(transaction_date)=datetime.date(2026, 4, 24))


In [0]:
gold_daily_demand.select(
    "daily_units_sold",
    "rolling_7d_units",
    "rolling_30d_units",
    "avg_daily_demand_7d",
    "avg_daily_demand_30d"
).describe().show()

+-------+--------------------+-------------------+-------------------+--------------------+--------------------+
|summary|    daily_units_sold|   rolling_7d_units|  rolling_30d_units| avg_daily_demand_7d|avg_daily_demand_30d|
+-------+--------------------+-------------------+-------------------+--------------------+--------------------+
|  count|            16546616|           16546616|           16546616|            16546616|            16546616|
|   mean|0.007070630030938048|0.04926922822164967|0.20582867215870604|0.007038461174464972|0.006860955738949615|
| stddev| 0.09416633334163933|0.26070261420893287| 0.5911719273419593| 0.03724323060127616|0.019705730911398663|
|    min|                -2.0|               -2.0|               -2.0| -0.2857142857142857|-0.06666666666666667|
|    max|                14.0|               15.0|               24.0|   2.142857142857143|                 0.8|
+-------+--------------------+-------------------+-------------------+--------------------+-----

In [0]:
gold_daily_demand.groupBy(
    "transaction_date",
    "product_id",
    "store"
).count().filter(
    F.col("count") > 1
).show()

+----------------+----------+-----+-----+
|transaction_date|product_id|store|count|
+----------------+----------+-----+-----+
+----------------+----------+-----+-----+



In [0]:
gold_daily_demand.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_demand")

print("Gold Daily Demand table created successfully.")

Gold Daily Demand table created successfully.


In [0]:
print(
    "Gold Daily Demand:",
    spark.table("workspace.default.gold_daily_demand").count()
)

Gold Daily Demand: 16546616


In [0]:
inventory_silver = spark.table("workspace.default.silver_inventory")

print("Inventory rows:", inventory_silver.count())

Inventory rows: 284755


In [0]:
inventory_silver.filter(
    (F.col("product_id") == "PROD-100805") &
    (F.col("store") == "STR-1006")
).orderBy(
    "start_date"
).show(100, False)

+----------+----------+------------+-----------+-----------+-------------------------------------------------------------+----------------+-------------------------------+--------------------------------------+-------------------------------------------------+--------+-----------------+-------------+-----------+--------------------+--------------+------------------+---------------+----------+
|start_date|end_date  |stock_status|supplier   |product_id |product_description                                          |product_division|product_category               |product_subcategory                   |product_segment                                  |store   |store_type       |sales_channel|qty_on_hand|stock_selling_amount|cost_of_stocks|unit_selling_price|unit_cost_price|is_current|
+----------+----------+------------+-----------+-----------+-------------------------------------------------------------+----------------+-------------------------------+-------------------------------------

In [0]:
inventory_silver.alias("a").join(
    inventory_silver.alias("b"),
    (F.col("a.product_id") == F.col("b.product_id")) &
    (F.col("a.store") == F.col("b.store")) &
    (F.col("a.start_date") < F.coalesce(
        F.col("b.end_date"),
        F.to_date(F.lit("9999-12-31"))
    )) &
    (F.col("b.start_date") < F.coalesce(
        F.col("a.end_date"),
        F.to_date(F.lit("9999-12-31"))
    )) &
    (F.col("a.start_date") != F.col("b.start_date")),
    "inner"
).select(
    F.col("a.product_id"),
    F.col("a.store"),
    F.col("a.start_date").alias("a_start"),
    F.col("a.end_date").alias("a_end"),
    F.col("b.start_date").alias("b_start"),
    F.col("b.end_date").alias("b_end")
).limit(20).show(20, False)

+----------+-----+-------+-----+-------+-----+
|product_id|store|a_start|a_end|b_start|b_end|
+----------+-----+-------+-----+-------+-----+
+----------+-----+-------+-----+-------+-----+

